# 05 · Dynamic Hydrographs

This notebook generates interactive Plotly hydrographs for each catchment in the study and saves them as standalone `.html` files in `results/hydrographs/`. Each file includes:

- Observed streamflow
- TFT median prediction
- Full quantile envelope (1–99%, 5–95%, 10–90%, 25–75%, 40–60%)
- HBV deterministic prediction
- Precipitation as a secondary inverted axis
- Air temperature as a third trace

Open any `.html` file directly in a browser — no server or Python installation required.

**Prerequisites:** pre-computed model outputs (Pickle files, `.p`) must be available in `results/model_outputs/`. These are generated by notebooks `02` and `03`.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUTPUTS_DIR = Path("../results/model_outputs")
HYDRO_DIR   = Path("../results/hydrographs")
(HYDRO_DIR / "ungauged").mkdir(parents=True, exist_ok=True)
(HYDRO_DIR / "specialized").mkdir(parents=True, exist_ok=True)

# Quantile bands to plot (lower, upper)
BANDS = [
    (0.01, 0.99),
    (0.05, 0.95),
    (0.10, 0.90),
    (0.25, 0.75),
    (0.40, 0.60),
]
BAND_COLORS = [
    "rgba(43,127,212,0.12)",
    "rgba(43,127,212,0.18)",
    "rgba(43,127,212,0.25)",
    "rgba(43,127,212,0.35)",
    "rgba(43,127,212,0.50)",
]
BAND_NAMES = ["1–99%", "5–95%", "10–90%", "25–75%", "40–60%"]

print("Setup complete.")

## Helper functions

In [ ]:
def load_station_data(station: str, setting: str) -> dict:
    """
    Load pre-computed model outputs for a given station.

    Parameters
    ----------
    station : str
        SNIRH station code (e.g., '03J01H').
    setting : str
        Either 'ungauged' or 'specialized'.

    Returns
    -------
    dict with keys: 'dates', 'observed', 'quantiles', 'hbv', 'precip', 'temp'
    """
    path = OUTPUTS_DIR / setting / f"{station}.p"
    if not path.exists():
        raise FileNotFoundError(f"No output file found: {path}")
    with open(path, "rb") as f:
        return pickle.load(f)


def build_hydrograph(
    data: dict,
    station: str,
    setting: str,
    nse: float | None = None,
    kge: float | None = None,
) -> go.Figure:
    """
    Build a Plotly interactive hydrograph figure.

    Parameters
    ----------
    data : dict
        Output of load_station_data().
    station : str
        Station code (for title).
    setting : str
        'ungauged' or 'specialized' (for title).
    nse, kge : float, optional
        Performance metrics to display in the title.

    Returns
    -------
    go.Figure
    """
    dates = data["dates"]
    obs   = data["observed"]
    quant = data["quantiles"]   # dict: float -> array
    hbv   = data.get("hbv")
    precip = data.get("precip")

    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.75, 0.25],
        shared_xaxes=True,
        vertical_spacing=0.04,
    )

    # --- Quantile bands ---
    for (lo, hi), color, name in zip(BANDS, BAND_COLORS, BAND_NAMES):
        y_lo = quant[lo]
        y_hi = quant[hi]
        fig.add_trace(
            go.Scatter(
                x=list(dates) + list(dates[::-1]),
                y=list(y_hi) + list(y_lo[::-1]),
                fill="toself",
                fillcolor=color,
                line=dict(width=0),
                name=name,
                legendgroup="bands",
                showlegend=True,
            ),
            row=1, col=1,
        )

    # --- Median prediction ---
    fig.add_trace(
        go.Scatter(
            x=dates, y=quant[0.50],
            mode="lines",
            line=dict(color="#2B7FD4", width=1.5, dash="dash"),
            name="TFT median",
        ),
        row=1, col=1,
    )

    # --- Observed ---
    fig.add_trace(
        go.Scatter(
            x=dates, y=obs,
            mode="lines",
            line=dict(color="black", width=1.2),
            name="Observed",
        ),
        row=1, col=1,
    )

    # --- HBV (if available) ---
    if hbv is not None:
        fig.add_trace(
            go.Scatter(
                x=dates, y=hbv,
                mode="lines",
                line=dict(color="#E87722", width=1.2),
                name="HBV",
            ),
            row=1, col=1,
        )

    # --- Precipitation (inverted) ---
    if precip is not None:
        fig.add_trace(
            go.Bar(
                x=dates, y=precip,
                marker_color="steelblue",
                opacity=0.6,
                name="Precipitation (mm)",
            ),
            row=2, col=1,
        )

    metric_str = ""
    if nse is not None:
        metric_str += f"  |  NSE = {nse:.2f}"
    if kge is not None:
        metric_str += f"  |  KGE = {kge:.2f}"

    fig.update_layout(
        title=dict(
            text=f"Station {station} — TFT {setting.capitalize()}{metric_str}",
            font=dict(size=14),
        ),
        height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1),
        hovermode="x unified",
        template="plotly_white",
    )
    fig.update_yaxes(
        title_text=r"q (10³ m³ s⁻¹ km⁻²)", row=1, col=1
    )
    fig.update_yaxes(
        title_text="P (mm)", autorange="reversed", row=2, col=1
    )
    fig.update_xaxes(title_text="Date", row=2, col=1)

    return fig

## Generate and export all hydrographs

In [ ]:
metrics_df = pd.read_csv("../results/tables/ungauged_metrics.csv")

for setting in ["ungauged", "specialized"]:
    output_files = sorted((OUTPUTS_DIR / setting).glob("*.p")) if (OUTPUTS_DIR / setting).exists() else []
    print(f"\n--- {setting.upper()} — {len(output_files)} stations ---")

    for pkl_path in output_files:
        station = pkl_path.stem
        try:
            data = load_station_data(station, setting)
        except FileNotFoundError as e:
            print(f"  SKIP {station}: {e}")
            continue

        # Retrieve metrics if available
        row = metrics_df[
            (metrics_df["station"] == station) &
            (metrics_df["model"] == ("TFT_ungauged" if setting == "ungauged" else "TFT_specialized"))
        ]
        nse = float(row["NSE"].values[0]) if len(row) else None
        kge = float(row["KGE"].values[0]) if len(row) else None

        fig = build_hydrograph(data, station, setting, nse=nse, kge=kge)
        out_path = HYDRO_DIR / setting / f"{station}.html"
        fig.write_html(str(out_path), include_plotlyjs="cdn")
        print(f"  Saved: {out_path.name}  (NSE={nse:.2f}, KGE={kge:.2f})" if nse else f"  Saved: {out_path.name}")

print("\nDone. Open any .html file in a browser to explore the hydrograph interactively.")

## Preview a single hydrograph inline

In [ ]:
# Change station and setting as needed
PREVIEW_STATION = "03J01H"
PREVIEW_SETTING = "ungauged"

try:
    data = load_station_data(PREVIEW_STATION, PREVIEW_SETTING)
    fig = build_hydrograph(data, PREVIEW_STATION, PREVIEW_SETTING)
    fig.show()
except FileNotFoundError:
    print(f"No output file found for {PREVIEW_STATION} ({PREVIEW_SETTING}).")
    print("Run notebook 02 first to generate model outputs.")